In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association

file_path = "amz_uk_price_prediction_dataset.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "asaniczka/uk-optimal-product-price-prediction",
  file_path,
)

display(df)

In [ ]:
df.columns

In [ ]:
category_best_seller = pd.crosstab(df['category'], df['isBestSeller'])

In [ ]:
crosstab_prop_sorted = category_best_seller.sort_values(by=True, ascending=False)
display(crosstab_prop_sorted.head(10))

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(crosstab_prop_sorted)
print(f"Chi-square Statistic: {chi2:.2f}")
print(f"P-value: {p_value}")

Null Hypothesis : Rejected because P-value is 0.0
There is a relationship between the product category and being a best-seller.

In [ ]:
cramer_v = association(category_best_seller, method="cramer")
print(f"Cramér's V: {cramer_v:.4f}")
cramer_v < 0.1
cramer_v < 0.3

Strength: Moderate

In [ ]:
df_top_10 = df[df['category'].isin(df['category'].value_counts().nlargest(10).index)]
crosstab_top_10_counts = pd.crosstab(df_top_10['category'], df_top_10['isBestSeller'])
crosstab_top_10_counts.plot(kind='bar')
plt.title('Stack Bar (product Category - Best Seller)')
plt.xlabel('Product Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Is Best Seller?')
plt.show()

In [ ]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)

IQR = Q3 - Q1

bound1 = Q1 - 1.5 * IQR
bound2 = Q3 + 1.5 * IQR

df_no_outliers = df[(df['price'] >= bound1) & (df['price'] <= bound2)]

print(f"Original dataset size: {len(df)}")
print(f"Dataset size without price outliers: {len(df_no_outliers)}")

In [ ]:
top_20 = df_no_outliers['category'].value_counts().nlargest(20).index
df_filtered = df_no_outliers[df_no_outliers['category'].isin(top_20)]

plt.figure(figsize=(16, 8))
sns.violinplot(data=df_filtered, x='category', y='price', order=top_20, hue='category', legend=False)

plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
medians = df_no_outliers.groupby('category')['price'].median()
highest_cat = medians.idxmax()
display(medians.sort_values(ascending=False))

In [ ]:

top_10 = df_no_outliers['category'].value_counts().nlargest(10).index

avg_price_top10 = (
    df_no_outliers[df_no_outliers['category'].isin(top_10)]
    .groupby('category')['price']
    .mean()
    .reindex(top_10)
)

avg_price_top10.plot(kind='bar', figsize=(12, 6), color='steelblue', edgecolor='black')

plt.xlabel('Product Category')
plt.ylabel('Average Price')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
top_10 = df_no_outliers['category'].value_counts().nlargest(10).index
df_top_10 = df_no_outliers[df_no_outliers['category'].isin(top_10)]
df_rated = df_top_10[df_top_10['stars'] > 0]
plt.figure(figsize=(14, 6))
sns.boxplot(
    data=df_rated,
    x='category',
    y='stars',
    order=top_10,
    hue='category',
    legend=False
)

plt.xlabel('Product Category')
plt.ylabel('Rating (Stars)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
df_rated_all = df_no_outliers[df_no_outliers['stars'] > 0]

median_ratings = df_rated_all.groupby('category')['stars'].median()

top_cat = median_ratings.idxmax()
top_val = median_ratings.max()

display(median_ratings.sort_values(ascending=False).head(5))

In [ ]:
df_corr = df_no_outliers[df_no_outliers['stars'] > 0][['price', 'stars']].dropna()

pearson_corr, pearson_p = stats.pearsonr(df_corr['price'], df_corr['stars'])

spearman_corr, spearman_p = stats.spearmanr(df_corr['price'], df_corr['stars'])

for name, corr, p in [("Pearson", pearson_corr, pearson_p), ("Spearman", spearman_corr, spearman_p)]:
    strength = "weak" if abs(corr) < 0.3 else "moderate" if abs(corr) < 0.6 else "strong"
    direction = "positive" if corr > 0 else "negative"
    sig = "significant" if p < 0.05 else "NOT significant"
    print(f"\n{name}: {strength} {direction} correlation, statistically {sig} (p={'<0.001' if p < 0.001 else f'{p:.4f}'})")


In [ ]:
df_scatter = df_no_outliers[df_no_outliers['stars'] > 0]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_scatter.sample(5000, random_state=42),  # sample for performance
    x='stars',
    y='price',
    alpha=0.4,
    color='steelblue'
)
plt.xlabel('Rating')
plt.ylabel('Price')
plt.tight_layout()
plt.show()


In [ ]:
numerical_cols = df_no_outliers.select_dtypes(include='number')
corr_matrix = numerical_cols.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    center=0,
    square=True
)
plt.tight_layout()
plt.show()
